# ISYE 7406 Project - Data Prep
# Species Distribution Modeling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [2]:
# Load the data

DATA_DIR = Path(".")  # assumes notebook is in same folder as csv files

file_map = {
    "Red-tailed Hawk": "hawk_env.csv",
    "Western Meadowlark": "meadowlark_env.csv",
    "American Robin": "robin_env.csv",
    "Spotted Towhee": "towhee_env.csv",
    "Steller's Jay": "jay_env.csv",
}

dfs = {}

for species_name, filename in file_map.items():
    df = pd.read_csv(DATA_DIR / filename)
    dfs[species_name] = df
    print(f"{species_name}: {df.shape[0]} rows x {df.shape[1]} columns")

Red-tailed Hawk: 8053 rows x 24 columns
Western Meadowlark: 7974 rows x 24 columns
American Robin: 7977 rows x 24 columns
Spotted Towhee: 7519 rows x 24 columns
Steller's Jay: 7635 rows x 24 columns


In [ ]:
# Quick Data quality checks
summary_rows = []

for species_name, df in dfs.items():
    summary_rows.append({
        "species": species_name,
        "rows": len(df),
        "columns": df.shape[1],
        "presence_count": int((df["presence"] == 1).sum()),
        "absence_count": int((df["presence"] == 0).sum()),
        "presence_rate": round(df["presence"].mean(), 4),
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_summary = pd.DataFrame(summary_rows)
quality_summary

,species,rows,columns,presence_count,absence_count,presence_rate,missing_values_total,duplicate_rows
0,Red-tailed Hawk,8053,24,2172,5881,0.2697,0,1901
1,Western Meadowlark,7974,24,2081,5893,0.2610,0,2002
2,American Robin,7977,24,2143,5834,0.2686,0,1876
3,Spotted Towhee,7519,24,1640,5879,0.2181,0,1835
4,Steller's Jay,7635,24,1768,5867,0.2316,0,1898


In [ ]:
# Renaming the bioclimatic variables for better readability

bio_rename = {
    "wc2.1_2.5m_bio_1": "bio1_annual_mean_temp",
    "wc2.1_2.5m_bio_2": "bio2_mean_diurnal_range",
    "wc2.1_2.5m_bio_3": "bio3_isothermality",
    "wc2.1_2.5m_bio_4": "bio4_temp_seasonality",
    "wc2.1_2.5m_bio_5": "bio5_max_temp_warmest_month",
    "wc2.1_2.5m_bio_6": "bio6_min_temp_coldest_month",
    "wc2.1_2.5m_bio_7": "bio7_temp_annual_range",
    "wc2.1_2.5m_bio_8": "bio8_mean_temp_wettest_quarter",
    "wc2.1_2.5m_bio_9": "bio9_mean_temp_driest_quarter",
    "wc2.1_2.5m_bio_10": "bio10_mean_temp_warmest_quarter",
    "wc2.1_2.5m_bio_11": "bio11_mean_temp_coldest_quarter",
    "wc2.1_2.5m_bio_12": "bio12_annual_precip",
    "wc2.1_2.5m_bio_13": "bio13_precip_wettest_month",
    "wc2.1_2.5m_bio_14": "bio14_precip_driest_month",
    "wc2.1_2.5m_bio_15": "bio15_precip_seasonality",
    "wc2.1_2.5m_bio_16": "bio16_precip_wettest_quarter",
    "wc2.1_2.5m_bio_17": "bio17_precip_driest_quarter",
    "wc2.1_2.5m_bio_18": "bio18_precip_warmest_quarter",
    "wc2.1_2.5m_bio_19": "bio19_precip_coldest_quarter",
}

clean_dfs = {}

for species_name, df in dfs.items():
    clean_df = df.rename(columns=bio_rename).copy()
    
    # land cover is categorical 
    if "land_cover" in clean_df.columns:
        clean_df["land_cover"] = clean_df["land_cover"].astype("category")
    
    clean_dfs[species_name] = clean_df

# Check one cleaned dataset
clean_dfs["American Robin"].head()

,presence,longitude,latitude,bio1_annual_mean_temp,bio10_mean_temp_warmest_quarter,bio11_mean_temp_coldest_quarter,bio12_annual_precip,bio13_precip_wettest_month,bio14_precip_driest_month,bio15_precip_seasonality,bio16_precip_wettest_quarter,bio17_precip_driest_quarter,bio18_precip_warmest_quarter,bio19_precip_coldest_quarter,bio2_mean_diurnal_range,bio3_isothermality,bio4_temp_seasonality,bio5_max_temp_warmest_month,bio6_min_temp_coldest_month,bio7_temp_annual_range,bio8_mean_temp_wettest_quarter,bio9_mean_temp_driest_quarter,land_cover,species
0,1,-123.268200,44.590471,11.397000,18.230000,5.210667,1281,227,16,70.194374,626,76,79,588,11.668667,44.455448,533.880676,27.412001,1.164,26.248001,5.541333,17.977333,13,American Robin
1,1,-124.054012,44.729266,10.434167,14.670000,6.618333,1982,329,30,66.068573,933,133,134,875,8.056666,48.592682,329.624908,19.510000,2.930,16.580000,6.996667,14.281666,11,American Robin
2,1,-122.525128,45.021766,9.595333,16.131332,4.102666,2060,322,29,59.826477,922,153,160,857,9.846666,42.159042,497.636444,23.936001,0.580,23.356001,4.398000,15.769333,1,American Robin
3,1,-122.676210,45.523440,11.510166,18.436001,4.904666,1077,172,19,59.535892,490,87,90,453,10.617000,41.700706,557.238831,26.516001,1.056,25.460001,5.290667,18.392666,13,American Robin
4,1,-121.891401,45.672289,11.053667,18.572001,3.541333,1481,239,20,63.458344,689,104,104,644,10.323334,38.194958,615.502808,26.888000,-0.140,27.028000,4.198000,18.572001,11,American Robin


In [10]:
# combine into one dataset
combined_df = pd.concat(clean_dfs.values(), ignore_index=True)

print("Combined shape:", combined_df.shape)
combined_df.head()


Combined shape: (39158, 24)


,presence,longitude,latitude,bio1_annual_mean_temp,bio10_mean_temp_warmest_quarter,bio11_mean_temp_coldest_quarter,bio12_annual_precip,bio13_precip_wettest_month,bio14_precip_driest_month,bio15_precip_seasonality,bio16_precip_wettest_quarter,bio17_precip_driest_quarter,bio18_precip_warmest_quarter,bio19_precip_coldest_quarter,bio2_mean_diurnal_range,bio3_isothermality,bio4_temp_seasonality,bio5_max_temp_warmest_month,bio6_min_temp_coldest_month,bio7_temp_annual_range,bio8_mean_temp_wettest_quarter,bio9_mean_temp_driest_quarter,land_cover,species
0,1,-123.087988,45.507174,11.232333,18.115334,4.828000,1167,203,14,70.578972,572,70,74,541,11.769334,44.472996,545.565918,27.208000,0.744,26.464001,5.147333,17.966667,13,Red-tailed Hawk
1,1,-122.820754,45.714480,11.298667,17.830667,4.962000,1115,178,19,61.552025,513,86,90,476,10.912666,43.359291,529.610229,26.048000,0.880,25.168001,5.369333,17.790667,10,Red-tailed Hawk
2,1,-123.021792,45.104299,11.472333,18.188000,5.180000,995,175,12,69.182381,485,60,65,452,11.798000,44.934490,534.857239,27.348000,1.092,26.256001,5.542000,18.087334,12,Red-tailed Hawk
3,1,-122.753008,43.918044,11.280833,18.113333,5.183333,1336,207,19,59.164677,597,104,104,536,11.814333,44.268333,532.151978,27.604000,0.916,26.688000,5.488000,17.910000,9,Red-tailed Hawk
4,1,-117.275528,45.374019,5.531833,14.511333,-3.210667,406,53,20,27.340179,134,73,99,103,15.476334,43.546238,719.454285,26.488001,-9.052,35.540001,8.636000,14.085333,12,Red-tailed Hawk


In [9]:
combined_df["species"].value_counts()

species
Red-tailed Hawk       8053
American Robin        7977
Western Meadowlark    7974
Steller's Jay         7635
Spotted Towhee        7519
Name: count, dtype: int64

In [14]:
# Save cleaned files

output_dir = Path("cleaned_data")
output_dir.mkdir(exist_ok=True)

for species_name, df in clean_dfs.items():
    safe_name = (
        species_name.lower()
        .replace(" ", "_")
        .replace("-", "")
        .replace("'", "")
    )
    df.to_csv(output_dir / f"{safe_name}_clean.csv", index=False)

combined_df.to_csv(output_dir / "all_species_combined_clean.csv", index=False)

print("Saved cleaned files to:", output_dir.resolve())

Saved cleaned files to: /Users/wtung/Documents/ISyE 7406 - Computational Data Analytics/Projec/cleaned_data
